# Apache Iceberg con AWS Glue
## Funcionalidades básicas

## Introducción <a class="anchor" id="intro"></a>

En este notebook, revisaremos las funcionalidades básicas de Apache Iceberg.

Puntos clave de aprendizaje:

1. Crear tablas Iceberg
2. Leer tablas Iceberg con la API de DataFrame y Spark SQL
3. Actualizar registros individuales
4. Eliminar registros de una tabla Iceberg (GDPR)
5. Viaje en el tiempo y rollback de cambios
6. Evolución del esquema

Trabajarás con dos tablas: **prepared_customer** y **prepared_web_sales**, que son versiones modificadas de las tablas [TPC-DS](https://www.tpc.org/tpcds/). Ya precreamos ambas tablas como parte de la configuración del workshop. Estas tablas se usarán para cargar datos en las tablas Iceberg creadas más abajo.

## Configuración del notebook <a class="anchor" id="setup"></a>

### Configuraciones

La siguiente celda configura un nuevo Catálogo de Spark llamado **iceberg_catalog**, compatible con el Glue Data Catalog y con soporte para tablas Iceberg. La propiedad `spark.sql.catalog.<catalog_name>.warehouse` se utiliza para definir la ubicación predeterminada de bases de datos y tablas creadas dentro de **iceberg_catalog**.

**Acción requerida:** Actualiza la propiedad `spark.sql.catalog.iceberg_catalog.warehouse` y reemplaza el marcador **\<your-account-id\>** con el ID actual de tu cuenta.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 4

%%configure
{
"--conf":"spark.sql.catalog.iceberg_catalog.warehouse=s3://otfs-workshop-data-<your-account-id>/datasets/glue_iceberg/ --conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.iceberg_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.iceberg_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog",
"--datalake-formats":"iceberg",
"--enable-spark-ui": "true",
"--spark-event-logs-path": "s3://otfs-workshop-data-<your-account-id>/glue_temp/sparkHistoryLogs/"
}

### Crear la sesión interactiva de Glue

Al ejecutar la siguiente celda, iniciarás la sesión interactiva de Glue en tu cuenta de AWS. Puedes usar la magia `%stop_session` para finalizar la sesión manualmente. La creación de la sesión puede tardar varios segundos. Espera a que finalice la creación de la sesión antes de continuar.

In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

### Parámetros

En esta sección definimos algunas variables que vamos a usar a lo largo del notebook.

**Acción requerida:** Actualiza la variable **BUCKET_NAME** y reemplaza el marcador **\<your-account-id\>** con el ID actual de tu cuenta.

In [ ]:
#Modify this variable
BUCKET_NAME = "otfs-workshop-data-<your-account-id>"

# python configs - do not modify
SERVICE = 'glue'
TPC_DS_DATABASE = "tpcds"
TPC_DS_CUSTOMER_TABLE = "prepared_customer"
TPC_DS_SALES_TABLE = "prepared_web_sales"
ICEBERG_CATALOG="iceberg_catalog"
ICEBERG_DB = f"{SERVICE}_iceberg_db"
ICEBERG_CUSTOMER_TABLE = "customer_iceberg"
ICEBERG_SALES_TABLE = "web_sales_iceberg"
ICEBERG_CATALOG_PATH=f"datasets/{SERVICE}_iceberg"
ICEBERG_DATABASE_PATH=f"{ICEBERG_CATALOG_PATH}/{ICEBERG_DB}.db"
WAREHOUSE_PATH=f"s3://{BUCKET_NAME}/{ICEBERG_CATALOG_PATH}"

# sparkmagic SQL configs - do not modify
spark.conf.set('ICEBERG_CATALOG', ICEBERG_CATALOG)
spark.conf.set('ICEBERG_DB', ICEBERG_DB)
spark.conf.set('ICEBERG_CUSTOMER_TABLE', ICEBERG_CUSTOMER_TABLE)
spark.conf.set('TPC_DS_DATABASE', TPC_DS_DATABASE)
spark.conf.set('TPC_DS_CUSTOMER_TABLE', TPC_DS_CUSTOMER_TABLE)

### Librerías y utilidades

En esta sección, importamos algunas librerías que vamos a usar a lo largo del notebook. Además, configuramos algunas funciones utilitarias que se usarán para mostrar las carpetas y archivos en S3 que forman parte de una tabla Iceberg dada.

In [ ]:
import boto3
import json
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col
from datetime import datetime, timedelta

s3_client = boto3.client('s3')
current_region = s3_client.meta.region_name
glue_client = boto3.client('glue', current_region)

def list_files_recursively(bucket_name, prefix):
    paginator = s3_client.get_paginator('list_objects_v2')
    operation_parameters = {'Bucket': bucket_name, 'Prefix': prefix}

    files_dict = {}
    for page in paginator.paginate(**operation_parameters):
        if 'Contents' in page:
            for obj in page['Contents']:
                file_key = obj['Key']
                if file_key.endswith('/'):  # It's a directory
                    relative_key = file_key[len(prefix):]  # Removing the prefix
                    if relative_key:  # Check if not an empty string
                        folders = relative_key.split('/')[:-1]  # Remove the last empty string element
                        current_dict = files_dict
                        for folder in folders:
                            if folder not in current_dict:
                                current_dict[folder] = {}
                            current_dict = current_dict[folder]
                else:  # It's a file
                    relative_key = file_key[len(prefix):]  # Removing the prefix
                    if '/' in relative_key:  # It's a file in a subdirectory
                        parent_folder, file_name = relative_key.rsplit('/', 1)
                        current_dict = files_dict
                        for folder in parent_folder.split('/'):
                            if folder:
                                if folder not in current_dict:  # Check if the key exists
                                    current_dict[folder] = {}
                                current_dict = current_dict[folder]
                        current_dict[file_name] = obj['LastModified'].strftime("%Y-%m-%d %H:%M:%S")
                    else:  # It's a file in the root directory
                        current_dict = files_dict
                        if 'data' not in current_dict:  # Check if the key exists
                            current_dict['data'] = {}
                        current_dict = current_dict['data']
                        files_dict[relative_key] = obj['LastModified'].strftime("%Y-%m-%d %H:%M:%S")

    return files_dict   

def print_files_tree(files_dict, indent=0):
    for key, value in files_dict.items():
        if isinstance(value, dict):  # It's a folder
            print("  " * indent + "- " + key)
            print_files_tree(value, indent + 1)
        else:  # When encountering the first file order the dict by ModifiedTime
            sorted_files = (sorted(files_dict.items(), key=lambda item: item[1]))
            for file_name, last_modified in sorted_files:
                print("  " * indent + "--- " + file_name + " (" + last_modified + ")")
            break
            

def show_tables_files(table_name):
    ICEBERG_TABLE_PATH = f"{ICEBERG_DATABASE_PATH}/{table_name}"
    print(f"TABLE ROOT LOCATION:\n s3://{BUCKET_NAME}/{ICEBERG_TABLE_PATH}/ \n\n")
    print("FOLDERS AND FILES:\n")
    files_dict = list_files_recursively(BUCKET_NAME, ICEBERG_TABLE_PATH)
    print_files_tree(files_dict)
    
def show_data_files(table_name):
    ICEBERG_TABLE_PATH = f"{ICEBERG_DATABASE_PATH}/{table_name}"
    prefix=f"{ICEBERG_TABLE_PATH}/data"
    print("FOLDERS AND FILES:\n")
    files_dict = list_files_recursively(BUCKET_NAME, prefix)
    print_files_tree(files_dict)
    
    
def show_tables_files_partition(partition, table_name):
    ICEBERG_TABLE_PATH = f"{ICEBERG_DATABASE_PATH}/{table_name}"
    prefix=f"{ICEBERG_TABLE_PATH}/data/{partition}"
    print(f"PARTITION LOCATION:\n s3://{BUCKET_NAME}/{prefix}/ \n\n")
    print("FOLDERS AND FILES:\n")
    files_dict = list_files_recursively(BUCKET_NAME, prefix)
    print_files_tree(files_dict)
    

def get_current_metadata_pointer(table_name):
    response = glue_client.get_table(
        DatabaseName=ICEBERG_DB,
        Name=table_name,
    )
    
    return response["Table"]["Parameters"]["metadata_location"]

def show_current_metadata_pointer(table_name):
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    print(current_metadata_pointer)
    
from urllib.parse import urlparse

def get_s3_file_content(s3_path):
    parsed_s3_path = urlparse(s3_path)
    bucket_name = parsed_s3_path.netloc
    object_key = parsed_s3_path.path.lstrip('/')
    try:
        response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
        file_content = response['Body'].read().decode('utf-8')
        return file_content
    except Exception as e:
        print(f"Error: {e}")
        return ""
    
    
def print_s3_file_content(s3_path):
    file_content = get_s3_file_content(s3_path)
    print(file_content)

        
def show_current_metadata_file(table_name):
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    print_s3_file_content(current_metadata_pointer)

def get_current_snapshot_manifest_list_file_name(table_name):
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    file_content = get_s3_file_content(current_metadata_pointer)
    json_file = json.loads(file_content)
    current_snapshot_id=json_file["current-snapshot-id"]
    for snapshot in json_file["snapshots"]:
        if snapshot["snapshot-id"] == current_snapshot_id:
            snapshot_manifest_list_file = snapshot["manifest-list"]
            return snapshot_manifest_list_file

def get_current_snapshot_manifest_list(table_name):
    current_manifest_list=get_current_snapshot_manifest_list_file_name(table_name)
    print(f"MANIFEST LIST FILE:\n{current_manifest_list}\n")
    df_manifest_list = spark.read.format("avro").load(current_manifest_list)
    return df_manifest_list

def remove_prefix(text, prefix):
    if text.startswith(prefix):
        return text[len(prefix):]
    return text

def s3_path_without_prefix(s3_path, table_name):
    ICEBERG_TABLE_PATH = f"{ICEBERG_DATABASE_PATH}/{table_name}"
    prefix = f"s3://{BUCKET_NAME}/{ICEBERG_TABLE_PATH}/"
    return remove_prefix(s3_path,prefix)


def print_snapshot_files_summary(snapshot_id, table_name):
    print(f"\n====== SNAPSHOT {snapshot_id} ======\n")
    print("DataFiles status: 0: EXISTING - 1: ADDED - 2: DELETED\n")
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    file_content = get_s3_file_content(current_metadata_pointer)
    json_file = json.loads(file_content)
    snapshot_manifest_list_file=""
    for snapshot in json_file["snapshots"]:
        if snapshot["snapshot-id"] == snapshot_id:
            snapshot_manifest_list_file = snapshot["manifest-list"]
            print(f"[MANIFEST LIST FILE]: {s3_path_without_prefix(snapshot_manifest_list_file, table_name)}")
            break
    df_manifest_list = spark.read.format("avro").load(snapshot_manifest_list_file)
    manifest_files=df_manifest_list.select("manifest_path", "added_snapshot_id").collect()
    for mf in manifest_files:
        manifest_file=mf["manifest_path"]
        added_snapshot_id=mf["added_snapshot_id"]
        print(f"\n--[MANIFEST FILE]: {s3_path_without_prefix(manifest_file, table_name)} [ADDED BY SNAPSHOT: {added_snapshot_id}]")
        df_manifest_file = spark.read.format("avro").load(manifest_file)
        data_files=df_manifest_file.select("data_file", "status").collect()
        for df in data_files:
            file_path=df["data_file"]["file_path"]
            status=df["status"]
            print(f"   |-[DATA FILE]: {s3_path_without_prefix(file_path, table_name)} (Status: {status})")
            
            
def print_snapshots_files_summary(table_name):
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    file_content = get_s3_file_content(current_metadata_pointer)
    json_file = json.loads(file_content)
    for snapshot in json_file["snapshots"]:
        print_snapshot_files_summary(snapshot["snapshot-id"])
        
def print_current_snapshot_files_summary(table_name):
    current_metadata_pointer=get_current_metadata_pointer(table_name)
    file_content = get_s3_file_content(current_metadata_pointer)
    json_file = json.loads(file_content)
    print_snapshot_files_summary(json_file["current-snapshot-id"], table_name)

## Catálogos <a class="anchor" id="catalogs"></a>

El catálogo predeterminado en Spark se llama **spark_catalog**.

In [ ]:
%%sql
SELECT current_catalog()

Cambiemos a **iceberg_catalog**, que configuramos arriba para las operaciones con Iceberg.

In [ ]:
%%sql
USE iceberg_catalog

Verifica que `current_catalog` sea **iceberg_catalog**

In [ ]:
%%sql
SELECT current_catalog()

Puedes acceder a bases de datos y tablas desde otro catálogo (que no sea el actual) usando el espacio de nombres completamente calificado: **\<catalog_id\>.\<database_name\>\.\<table_name\>**

In [ ]:
%%sql
SHOW TABLES IN spark_catalog.${TPC_DS_DATABASE}

## Creación de base de datos <a class="anchor" id="db_creation"></a>

Creemos nuestra primera base de datos en el catálogo Iceberg.

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS ${ICEBERG_DB}

## Creación de tablas <a class="anchor" id="table_creation"></a>

Después de crear la base de datos, podemos empezar creando una tabla con datos de la tabla de clientes en bruto. Usaremos esta tabla para explorar la funcionalidad básica de Iceberg.

In [ ]:
%%sql
CREATE TABLE ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} (
    c_customer_sk INT COMMENT 'unique id', 
    c_customer_id STRING, 
    c_first_name STRING, 
    c_last_name STRING, 
    c_email_address STRING
) USING iceberg
TBLPROPERTIES (
    'format-version'='2'
)
COMMENT 'This table contains customer data'

Puedes verificar la nueva tabla en la base de datos con el comando de abajo, pero ten en cuenta que, por el momento, no hay datos en la tabla.

In [ ]:
%%sql
SHOW TABLES IN ${ICEBERG_DB}

Puedes revisar información detallada de la tabla usando la consulta DESCRIBE.

In [ ]:
spark.sql(f"""
DESCRIBE TABLE EXTENDED {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}
"""
).show(truncate=False)

También puedes revisar las propiedades de la tabla con SHOW TBLPROPERTIES.

In [ ]:
spark.sql(f"""
SHOW TBLPROPERTIES {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}
"""
).show(truncate=False)

## Estructura de una tabla Iceberg <a class="anchor" id="table_structure"></a>

Aquí hay un diagrama que representa la estructura subyacente de una tabla Iceberg:

- cada operación de commit (insert, update, delete, merge, compaction) en Iceberg genera una nueva instantánea (snapshot) (por ejemplo, S0, S1, ...)
- cada operación (commit, actualización de esquema, rollback, roll-forward) en Iceberg genera un nuevo archivo de metadatos
- la tabla en el catálogo de Iceberg apunta al archivo de metadatos más reciente (puntero de metadatos actual)
- el archivo de metadatos, entre otra información, contiene una lista de instantáneas disponibles y el ID de la instantánea actual
- cada instantánea de la tabla está asociada a un archivo de lista de manifiestos (manifest list)
- cada archivo de lista de manifiestos, entre otra información, apunta a uno o varios archivos de manifiesto (manifest)
- cada archivo de manifiesto apunta a uno o varios archivos de datos

<div>
<center><img src="https://aws-data-analytics-workshops.s3.amazonaws.com/transactional-and-mutable-datalakes/misc/images/iceberg-metadata.png" width="700"/></center>
</div>

La tabla de clientes no contiene datos por el momento. Por lo tanto, solo hay **un archivo de metadatos** en la carpeta **metadata**. Verás una carpeta **data** que contendrá archivos de datos una vez ingiramos datos.

In [ ]:
show_tables_files(ICEBERG_CUSTOMER_TABLE)

El Glue Data Catalog mantiene un puntero al archivo de metadatos más reciente, ***metadata.json**.

In [ ]:
show_current_metadata_pointer(ICEBERG_CUSTOMER_TABLE)

A continuación se muestra el contenido del archivo de metadatos. Almacena el esquema y las propiedades de la tabla. Como puedes ver, por ahora no tenemos instantáneas (el campo **snapshots** es una lista vacía):

In [ ]:
show_current_metadata_file(ICEBERG_CUSTOMER_TABLE)

## Insertar datos <a class="anchor" id="insert"></a>

Ahora inserta algunos datos de clientes en la tabla Iceberg. Aquí hay una muestra de la tabla fuente.

In [ ]:
%%sql
SELECT *
FROM spark_catalog.${TPC_DS_DATABASE}.${TPC_DS_CUSTOMER_TABLE}
LIMIT 10

Vamos a insertarlos

In [ ]:
%%sql
INSERT INTO ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE}
SELECT *
FROM spark_catalog.${TPC_DS_DATABASE}.${TPC_DS_CUSTOMER_TABLE}

Puedes ver que se insertaron 2 millones de registros de clientes en la tabla.

In [ ]:
%%sql
SELECT COUNT(*)
FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE}

Ahora volvamos a imprimir la estructura de la tabla.

Verás dos carpetas, **data** y **metadata**, donde la carpeta **data** contiene los datos reales en formato Parquet y la carpeta **metadata** tiene varios archivos de metadatos.

Hay tres tipos de archivos de metadatos:
1) archivos de metadatos, que terminan en **.metadata.json**
2) archivos de lista de manifiestos (manifest list), con el formato **snap-*.avro**
3) archivos de manifiesto (manifest), que terminan en **-m*\.avro**.

Se creará un nuevo archivo de metadatos cada vez que hagas cambios en la tabla. No necesitas entender los detalles de los archivos de lista de manifiestos o de manifiesto para el propósito de este laboratorio.

In [ ]:
show_tables_files(ICEBERG_CUSTOMER_TABLE)

Puedes ver que el puntero de metadatos actual en la tabla del Glue Data Catalog se actualizó para apuntar al nuevo archivo de metadatos.

In [ ]:
show_current_metadata_pointer(ICEBERG_CUSTOMER_TABLE)

## Consultar la tabla Iceberg

Ahora tienes tu primera tabla Iceberg. Veamos cómo consultar datos de esta tabla.

Iceberg soporta consultas con Spark SQL y con DataFrames. Verás que estas dos APIs se usan de forma intercambiable en este laboratorio.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} LIMIT 10

In [ ]:
# Querying with DataFrame
spark.table(f"{ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}").limit(10).show()

## Actualizar registros

Tu siguiente tarea es hacer una limpieza de datos. En la siguiente sección, te enfocarás en una cliente específica que ingresó su apellido y correo electrónico de forma incorrecta. Como resultado, esos dos campos son *null* y debes corregirlos.

Puedes hacer ese cambio fácilmente usando la consulta `UPDATE`. Las consultas UPDATE aceptan un filtro para seleccionar las filas a actualizar.

Observa que el apellido y la dirección de correo de Tonya son *null*. Tu equipo de servicio al cliente recopiló su apellido y correo y te los proporcionó.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

In [ ]:
%%sql
UPDATE ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} SET c_last_name = 'John', c_email_address = 'johnTonya@abx.com' WHERE c_customer_sk = 15

Observa que el apellido y la dirección de correo de Tonya quedaron corregidos.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

Puedes revisar cómo esta operación UPDATE impacta la capa de datos de Iceberg. Como el laboratorio usa el formato de cambio predeterminado Copy-On-Write, cuando se realiza un cambio para eliminar o actualizar una o varias filas, los archivos de datos que contienen esas filas se duplican, y la nueva versión incluye las filas actualizadas.

Observa que se creó un nuevo archivo Parquet debido al cambio anterior. Puedes identificar ese archivo por tener un timestamp de LastModified diferente. Esto representa un estilo de cambio Copy-On-Write.

In [ ]:
show_data_files(ICEBERG_CUSTOMER_TABLE)

## Eliminar registros

Tonya decidió retirarse de la aplicación ejerciendo sus derechos bajo GDPR. Ahora necesitas eliminar sus registros.

Puedes hacer ese cambio fácilmente usando la consulta `DELETE FROM`. Las consultas DELETE FROM aceptan un filtro para seleccionar las filas a eliminar.

In [ ]:
%%sql
DELETE FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

Observa que el registro de Tonya fue eliminado de la tabla Iceberg.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

De nuevo, revisemos cómo esto impacta la capa de datos de Iceberg. Si el filtro de borrado coincide con particiones completas de la tabla, Iceberg realizará un borrado solo a nivel de metadatos. Si el filtro coincide con filas individuales de una tabla, entonces Iceberg reescribirá únicamente los archivos de datos afectados. De nuevo, como el laboratorio usa el formato predeterminado Copy-On-Write a nivel de fila, cuando se realiza un cambio para eliminar o actualizar una o varias filas, los archivos de datos que contienen esas filas se duplican, pero la nueva versión incluye las filas actualizadas.

Observa que se creó un nuevo archivo Parquet debido al cambio anterior. Puedes identificar ese archivo por tener un timestamp de LastModified diferente. Esto representa un estilo de cambio Copy-On-Write.

In [ ]:
show_data_files(ICEBERG_CUSTOMER_TABLE)

## Viaje en el tiempo y rollback

El viaje en el tiempo (time-travel) habilita consultas reproducibles apuntando a una instantánea específica de la tabla y permite examinar cambios fácilmente. El rollback de versiones permite corregir problemas rápidamente restableciendo tablas a un estado bueno. En la siguiente sección, revisarás el historial de la tabla de clientes para los datos de Tonya y harás rollback de la eliminación de su registro.

Cada cambio en una tabla Iceberg crea una versión independiente del árbol de metadatos, llamada instantánea (snapshot). Verás tres instantáneas en total usando la siguiente consulta:
1. operación inicial de inserción (append)
2. operación de actualización (overwrite)
3. operación de borrado (overwrite)

In [ ]:
spark.sql(f"""
SELECT * FROM {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}.snapshots
ORDER BY committed_at
"""
).show(truncate=False, vertical=True)

Observa que la consulta anterior está apuntando a la tabla de metadatos **snapshots**. Iceberg expone varios metadatos de tabla (por ejemplo, historial, estadísticas de particiones y estadísticas de instantáneas) a través de tablas de metadatos. Las tablas de metadatos se identifican agregando el nombre de la tabla de metadatos después del nombre de la tabla original. Discutiremos los metadatos de tabla con más detalle en el siguiente laboratorio.

Aquí podemos revisar el historial de acciones realizadas sobre nuestra tabla.

In [ ]:
spark.sql(f"""
SELECT * FROM {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}.history
ORDER BY made_current_at
"""
).show(truncate=False)

Tomemos los snapshot IDs de las últimas 2 operaciones sobre la tabla para hacer viaje en el tiempo y rollback.

In [ ]:
snapshotIDs = spark.sql(f"""
SELECT snapshot_id
FROM {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}.snapshots
ORDER BY committed_at DESC
"""
).collect()

second_snapshot_id = snapshotIDs[1][0]
latest_snapshot_id = snapshotIDs[0][0]

print(f"second_snapshot_id: {second_snapshot_id}")
print(f"latest_snapshot_id: {latest_snapshot_id}")

Puedes leer un estado específico de la tabla indicando un snapshot-id o un timestamp. La siguiente consulta realiza un viaje en el tiempo a la versión anterior a la eliminación del registro de Tonya.

In [ ]:
query = f"""
SELECT * FROM {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE} FOR VERSION AS OF {second_snapshot_id} where c_customer_sk = 15
"""
spark.sql(query).show() 

El viaje en el tiempo permite volver atrás y revisar el estado de una tabla en el pasado. Iceberg proporciona una forma de hacer rollback de cambios y llevar permanentemente la tabla a un estado histórico usando el snapshot_id de una instantánea anterior. Esto es útil para revertir eliminaciones accidentales o corrupción de datos. Para hacerlo, usa la sentencia `CALL` e invoca el procedimiento almacenado de Iceberg `rollback_to_snapshot` para volver a cualquier commit histórico.

La siguiente consulta hace rollback de una tabla a un snapshot ID específico, revirtiendo así la eliminación del registro de Tonya.

In [ ]:
query = f"""
CALL system.rollback_to_snapshot('{ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}', {second_snapshot_id})
"""
spark.sql(query).show() 

Observa que el registro de Tonya vuelve a estar presente en la versión actual de la tabla.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

Ten en cuenta que cada operación de rollback o roll-forward genera un nuevo archivo de metadatos. Pero no impacta los archivos de datos.

In [ ]:
show_data_files(ICEBERG_CUSTOMER_TABLE)

Verás una nueva versión de instantánea agregada al final de la tabla de metadatos de historial. También notarás que la versión anterior quedó marcada como NOT current ancestor debido al rollback.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE}.history
ORDER BY made_current_at

También puedes hacer roll-forward al último estado de la tabla (después de la operación de BORRADO).

In [ ]:
query = f"""
CALL system.set_current_snapshot('{ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE}', {latest_snapshot_id})
"""
spark.sql(query).show() 

Observa que el registro de Tonya se elimina nuevamente.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} WHERE c_customer_sk = 15

Como puedes ver en la tabla de historial, la tercera instantánea ahora tiene `is_current_ancestor` revertido a true. Además, puedes ver una nueva operación agregada.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE}.history
ORDER BY made_current_at

De nuevo, cada operación de rollback y roll-forward genera un nuevo archivo de metadatos (ver archivos json en la carpeta metadata), pero no genera nuevas instantáneas (no hay nuevos archivos de lista de manifiestos ni archivos de datos).

In [ ]:
show_tables_files(ICEBERG_CUSTOMER_TABLE)

## Evolución del esquema

¿Qué pasa si necesitas renombrar una columna, eliminar una columna o agregar nuevas columnas? Tradicionalmente esto es una tarea dolorosa con tablas Hive, porque necesitas sobrescribir los datos existentes. Con Iceberg, la evolución del esquema simplemente funciona. Agregar nuevas columnas no impacta los datos existentes. Las columnas pueden agregarse, eliminarse, renombrarse y reordenarse. Lo mejor es que los cambios de esquema nunca requieren reescribir tu tabla. Consulta la [documentación](https://iceberg.apache.org/docs/latest/evolution/) para más detalles.

Al revisar el archivo de metadatos actual, podemos ver que solo tenemos un esquema (campo **schemas**) que es el definido durante la creación de la tabla.

In [ ]:
show_current_metadata_file(ICEBERG_CUSTOMER_TABLE)

Puedes usar el siguiente comando DDL para cambiar el nombre de la columna *c_email_address* a *email*.

In [ ]:
%%sql
ALTER TABLE ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} RENAME COLUMN c_email_address TO email

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} limit 10

Observa que no se creó ningún archivo de datos nuevo debido a la evolución del esquema. El cambio de esquema se almacena en la capa de metadatos.

In [ ]:
show_data_files(ICEBERG_CUSTOMER_TABLE)

Como puedes ver en el último archivo de metadatos de abajo, ahora tienes 2 esquemas: schema-id=0 (el original) y schema-id=1 (el nuevo). Observa que los valores de **name** son diferentes para la columna con **"id" : 5**. Iceberg utiliza mapeo de esquemas para que la evolución del esquema sea ligera y flexible.

In [ ]:
show_current_metadata_file(ICEBERG_CUSTOMER_TABLE)

Puedes usar el siguiente comando DDL para agregar una nueva columna.

In [ ]:
%%sql
ALTER TABLE ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} ADD COLUMN c_birth_date int

Observa que la nueva columna, por defecto, tiene **null** como valor para todas las filas existentes.

In [ ]:
%%sql
SELECT * FROM ${ICEBERG_DB}.${ICEBERG_CUSTOMER_TABLE} limit 10

Las pruebas anteriores aumentan la confianza de tu equipo en Iceberg. Ahora quieres continuar la prueba con la tabla de ventas, que contiene 7 millones de eventos de ventas, para validar la eficiencia de Iceberg.

## Limpieza [NO EJECUTAR a menos que quieras recrear los datasets] <a class="anchor" id="cleanup"></a>

Los comandos de abajo eliminan la base de datos y la tabla Iceberg creadas del catálogo, así como los archivos relacionados en S3.

In [ ]:
#Remove the created Iceberg table
spark.sql(f"DROP TABLE {ICEBERG_DB}.{ICEBERG_CUSTOMER_TABLE} PURGE")

In [ ]:
#Remove the created Iceberg database
spark.sql(f"DROP DATABASE {ICEBERG_DB}")